import json, re, time, html
from pathlib import Path
import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:3b"
SEED = 42

OUT_DIR = Path("results_v2")
OUT_DIR.mkdir(exist_ok=True)

# Ollama çalışıyor mu, model kurulu mu?
tags = requests.get("http://localhost:11434/api/tags", timeout=10).json()
print("Kurulu modeller:", [m["name"] for m in tags["models"]])

DATA_URL = "https://huggingface.co/datasets/osanseviero/twitter-airline-sentiment/resolve/main/Tweets.csv"

full = pd.read_csv(
    DATA_URL,
    usecols=["text", "airline_sentiment", "airline", "negativereason"]
).rename(columns={"airline_sentiment": "true_label"})

full["text"] = full["text"].apply(html.unescape)          # &amp; gibi kalıntıları temizle
full = full.drop_duplicates(subset="text").reset_index(drop=True)
print("Toplam tweet:", len(full))
print("Havayolları:", full["airline"].unique().tolist())

# Issue etiketleri: veri setindeki negativereason değerleri ("Can't Tell" yerine "Other")
ISSUE_LABELS = sorted(l for l in full["negativereason"].dropna().unique() if l != "Can't Tell") + ["Other"]
print("Issue etiketleri:", ISSUE_LABELS)

def balanced_sample(df, n_per_class, seed):
    parts = [df[df["true_label"] == c].sample(n_per_class, random_state=seed)
             for c in ["negative", "neutral", "positive"]]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

dev = balanced_sample(full, 50, SEED)                              # prompt geliştirme: 150 tweet
rest = full[~full["text"].isin(dev["text"])]
test = balanced_sample(rest, 100, SEED)                            # final değerlendirme: 300 tweet

print("\nDev:", dev.shape, "| Test:", test.shape)
print("Dev ∩ Test:", len(set(dev["text"]) & set(test["text"])))   # 0 olmalı
print("\nDev havayolu dağılımı:\n", dev["airline"].value_counts())

CATEGORIES = ["positive", "negative", "neutral"]
ENTITY_TYPES = ["airline", "person", "location", "flight_number", "organization", "other"]

# Ollama "structured outputs" için JSON şeması
JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": CATEGORIES},
        "sentiment_score": {"type": "number"},
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"},
                    "type": {"type": "string", "enum": ENTITY_TYPES}
                },
                "required": ["text", "type"]
            }
        },
        "issues": {"type": "array", "items": {"type": "string", "enum": ISSUE_LABELS}}
    },
    "required": ["category", "sentiment_score", "entities", "issues"]
}

SYSTEM_PROMPT = f"""You are an information extraction system for airline customer tweets.
Return ONLY one valid JSON object. No markdown, no explanations.

Output format:
{{"category": "positive|negative|neutral", "sentiment_score": 0.0, "entities": [{{"text": "...", "type": "..."}}], "issues": ["..."]}}

Field definitions:
- category: overall sentiment expressed by the customer about the airline or its service.
  positive = clear satisfaction, praise or gratitude.
  negative = clear complaint, frustration or a reported service problem.
  neutral = factual information, a plain question, news, or no clear opinion.
- sentiment_score: polarity from 0.0 (very negative) to 1.0 (very positive); 0.5 means neutral.
- entities: named entities that appear in the tweet. "text" must be copied exactly from the tweet.
  Airline names and @handles have type airline. "type" must be one of {ENTITY_TYPES}.
- issues: problems reported by the customer. Each item must be one of {ISSUE_LABELS}.
  Use [] when no problem is reported.

Rules:
- Judge the whole tweet, not single keywords.
- Mentioning an airline or a flight does not make a tweet negative.
- A question is neutral unless it clearly complains.
- Do not invent entities or issues that are not in the tweet.
- If there are no entities or issues, use empty lists.
"""

# Few-shot örnekleri tamamen uydurma (veri setinden alınmadı → sızıntı yok)
FEW_SHOT = [
    ("@SkyJet thank you for the smooth flight and the friendly crew!",
     {"category": "positive", "sentiment_score": 0.95,
      "entities": [{"text": "@SkyJet", "type": "airline"}], "issues": []}),
    ("@SkyJet my flight to Boston was delayed 3 hours and nobody told us why",
     {"category": "negative", "sentiment_score": 0.05,
      "entities": [{"text": "@SkyJet", "type": "airline"}, {"text": "Boston", "type": "location"}],
      "issues": ["Late Flight", "Customer Service Issue"]}),
    ("@SkyJet what time does flight 456 leave tomorrow?",
     {"category": "neutral", "sentiment_score": 0.5,
      "entities": [{"text": "@SkyJet", "type": "airline"}, {"text": "flight 456", "type": "flight_number"}],
      "issues": []}),
]

def build_prompt(tweet, few_shot=False):
    prompt = SYSTEM_PROMPT
    if few_shot:
        prompt += "\nExamples:\n"
        for t, out in FEW_SHOT:
            prompt += f"\nTweet: {json.dumps(t)}\nJSON: {json.dumps(out)}\n"
    prompt += f"\nTweet: {json.dumps(tweet, ensure_ascii=False)}\nJSON:"
    return prompt

print(build_prompt("@United my bag is lost again", few_shot=False)[-400:])

def parse_llm_json(raw):
    """
    Kademeli temizleme. Dönüş: (veri, aşama)
    aşama: 'raw' (doğrudan parse) | 'cleaned' (markdown/konuşma temizlendi)
           | 'repaired' (sondaki virgüller düzeltildi) | 'failed'
    """
    try:
        return json.loads(raw), "raw"
    except (json.JSONDecodeError, TypeError):
        pass

    try:
        text = re.sub(r"```(?:json)?", "", raw, flags=re.I).strip()   # markdown bloklarını sil
        start, end = text.find("{"), text.rfind("}")                   # konuşma metnini at
        if start != -1 and end > start:
            candidate = text[start:end + 1]
            try:
                return json.loads(candidate), "cleaned"
            except json.JSONDecodeError:
                fixed = re.sub(r",\s*([}\]])", r"\1", candidate)       # sondaki virgülleri düzelt
                return json.loads(fixed), "repaired"
    except (json.JSONDecodeError, TypeError, AttributeError):
        pass

    return None, "failed"


def validate_output(data):
    """Şema doğrulaması. Dönüş: (geçerli_mi, hata_mesajı)"""
    required = {"category", "sentiment_score", "entities", "issues"}
    if not isinstance(data, dict):
        return False, "not a JSON object"
    if set(data.keys()) != required:
        return False, f"keys mismatch: {sorted(data.keys())}"
    if data["category"] not in CATEGORIES:
        return False, f"invalid category: {data['category']}"
    s = data["sentiment_score"]
    if isinstance(s, bool) or not isinstance(s, (int, float)) or not 0 <= s <= 1:
        return False, f"invalid sentiment_score: {s}"
    if not isinstance(data["entities"], list) or not isinstance(data["issues"], list):
        return False, "entities/issues must be lists"
    for e in data["entities"]:
        if not (isinstance(e, dict) and isinstance(e.get("text"), str) and isinstance(e.get("type"), str)):
            return False, f"invalid entity: {e}"
    if not all(isinstance(i, str) for i in data["issues"]):
        return False, "issues must be strings"
    return True, ""

# Parser testi: bozuk çıktıları gerçekten yakalıyor mu?
tests = [
    '{"category": "positive", "sentiment_score": 0.9, "entities": [], "issues": []}',
    '```json\n{"category": "negative", "sentiment_score": 0.1, "entities": [], "issues": []}\n```',
    'Here is your JSON:\n{"category": "neutral", "sentiment_score": 0.5, "entities": [], "issues": []}\nHope this helps!',
    '{"category": "neutral", "sentiment_score": 0.5, "entities": [], "issues": [],}',
    'Sorry, I cannot do that.',
]
for t in tests:
    data, stage = parse_llm_json(t)
    print(f"{stage:9s} | valid={validate_output(data)[0] if data else False}")

def query_ollama(prompt, mode="schema", temperature=0.0, seed=SEED):
    """
    mode:
      'none'   -> format kısıtı yok (modelin kendi çıktısı)
      'json'   -> format='json' (geçerli JSON zorlanır)
      'schema' -> format=JSON_SCHEMA (yapı da zorlanır)
    """
    payload = {
        "model": MODEL_NAME, "prompt": prompt, "stream": False,
        "options": {"temperature": temperature, "seed": seed}
    }
    if mode == "json":
        payload["format"] = "json"
    elif mode == "schema":
        payload["format"] = JSON_SCHEMA

    t0 = time.time()
    r = requests.post(OLLAMA_URL, json=payload, timeout=180)
    r.raise_for_status()
    return r.json()["response"], time.time() - t0

def process_tweet(tweet, mode, few_shot=False, max_retries=1):
    prompt = build_prompt(tweet, few_shot)
    rec = {"first_raw_parse": False, "first_parse_after_cleaning": False,
           "first_schema_ok": False, "attempts": 0, "final_ok": False,
           "error": "", "raw_output": "", "latency": 0.0, "parsed": None}

    for attempt in range(max_retries + 1):
        rec["attempts"] = attempt + 1
        try:
            # İlk deneme deterministik; retry'da farklı çıktı alabilmek için sıcaklık artırılır
            raw, lat = query_ollama(prompt, mode,
                                    temperature=0.0 if attempt == 0 else 0.5,
                                    seed=SEED + attempt)
            rec["latency"] += lat
            rec["raw_output"] = raw

            data, stage = parse_llm_json(raw)
            ok, err = (False, "parse failed") if data is None else validate_output(data)

            if attempt == 0:   # ödevin "ilk denemede başarı" metrikleri
                rec["first_raw_parse"] = (stage == "raw") and isinstance(data, dict)
                rec["first_parse_after_cleaning"] = stage != "failed"
                rec["first_schema_ok"] = ok

            if ok:
                rec["final_ok"], rec["parsed"], rec["error"] = True, data, ""
                break
            rec["error"] = err

        except Exception as e:     # bağlantı hatası, timeout vb. programı çökertmez
            rec["error"] = f"{type(e).__name__}: {e}"
    return rec


def run_experiment(df, mode, few_shot=False, name="exp"):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=name):
        rec = process_tweet(row["text"], mode, few_shot)
        p = rec.pop("parsed")
        rec.update({
            "text": row["text"], "true_label": row["true_label"],
            "airline": row["airline"], "negativereason": row["negativereason"],
            "pred_label": p["category"] if p else None,
            "sentiment_score": p["sentiment_score"] if p else None,
            "entities": json.dumps(p["entities"], ensure_ascii=False) if p else None,
            "issues": json.dumps(p["issues"], ensure_ascii=False) if p else None,
        })
        rows.append(rec)
    res = pd.DataFrame(rows)
    res.to_csv(OUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    return res


def json_report(res, name):
    return {
        "experiment": name,
        "n": len(res),
        "raw_parse_%": round(res["first_raw_parse"].mean() * 100, 2),
        "parse_after_cleaning_%": round(res["first_parse_after_cleaning"].mean() * 100, 2),
        "schema_valid_first_try_%": round(res["first_schema_ok"].mean() * 100, 2),
        "final_after_retry_%": round(res["final_ok"].mean() * 100, 2),
        "retries_used": int((res["attempts"] > 1).sum()),
        "accuracy_%": round((res["pred_label"] == res["true_label"]).mean() * 100, 2),  # hatalar yanlış sayılır
        "avg_latency_s": round(res["latency"].mean(), 2),
    }

for mode in ["none", "json", "schema"]:
    print(f"\n===== MOD: {mode} =====")
    for t in dev["text"].head(3):
        raw, lat = query_ollama(build_prompt(t), mode)
        data, stage = parse_llm_json(raw)
        print(f"[{stage}] {lat:.1f}s | {raw[:150]!r}")

exp_none   = run_experiment(dev, "none",   name="dev_zero_shot_format_none")
exp_json   = run_experiment(dev, "json",   name="dev_zero_shot_format_json")
exp_schema = run_experiment(dev, "schema", name="dev_zero_shot_format_schema")

json_table = pd.DataFrame([
    json_report(exp_none,   "no format constraint"),
    json_report(exp_json,   "format='json'"),
    json_report(exp_schema, "format=JSON schema"),
])
display(json_table)
json_table.to_csv(OUT_DIR / "json_robustness_table.csv", index=False, encoding="utf-8-sig")

# Bozulan çıktılar neye benziyor? (raporda örnek olarak kullanılacak)
bad = exp_none[~exp_none["first_raw_parse"]]
print(f"\nFormat kısıtı olmadan ilk denemede ham parse başarısız: {len(bad)} / {len(exp_none)}")
for raw in bad["raw_output"].head(5):
    print("-", repr(raw[:200]))

WEAK_PROMPT = """Analyze the sentiment of this airline tweet. Tell me the sentiment (positive, negative or neutral), a score from 0 to 1, any named entities, and any problems the customer mentions. Give your answer in JSON.

Tweet: {tweet}"""

def stress_run(df, prompt_fn, temperature, name):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=name):
        rec = {"text": row["text"], "raw_parse": False, "after_cleaning": False,
               "schema_ok": False, "stage": "error", "error": "", "raw_output": ""}
        try:
            r = requests.post(OLLAMA_URL, json={
                "model": MODEL_NAME, "prompt": prompt_fn(row["text"]), "stream": False,
                "options": {"temperature": temperature, "seed": SEED, "num_predict": 500}
            }, timeout=180)
            r.raise_for_status()
            raw = r.json()["response"]
            data, stage = parse_llm_json(raw)
            ok, err = (False, "parse failed") if data is None else validate_output(data)
            rec.update({"raw_output": raw, "stage": stage,
                        "raw_parse": stage == "raw" and isinstance(data, dict),
                        "after_cleaning": stage != "failed",
                        "schema_ok": ok, "error": err})
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
        rows.append(rec)
    res = pd.DataFrame(rows)
    res.to_csv(OUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    return res

stress_conditions = {
    "S1: weak prompt, temp=0.0":       (lambda t: WEAK_PROMPT.format(tweet=t), 0.0),
    "S2: weak prompt, temp=1.0":       (lambda t: WEAK_PROMPT.format(tweet=t), 1.0),
    "S3: strong prompt, temp=1.5":     (lambda t: build_prompt(t), 1.5),
}

stress_rows, stress_results = [], {}
for i, (label, (fn, temp)) in enumerate(stress_conditions.items(), 1):
    res = stress_run(dev, fn, temp, f"stress_S{i}")
    stress_results[label] = res
    stress_rows.append({
        "condition": label,
        "raw_parse_%": round(res["raw_parse"].mean() * 100, 2),
        "parse_after_cleaning_%": round(res["after_cleaning"].mean() * 100, 2),
        "schema_valid_%": round(res["schema_ok"].mean() * 100, 2),
        "rescued_by_cleaning": int((res["after_cleaning"] & ~res["raw_parse"]).sum()),
    })

stress_table = pd.DataFrame(stress_rows)
display(stress_table)
stress_table.to_csv(OUT_DIR / "stress_test_table.csv", index=False, encoding="utf-8-sig")

# Bozulma örnekleri (rapor için)
res = stress_results["S2: weak prompt, temp=1.0"]
print("\n--- Temizleme ile kurtarılan örnekler ---")
for raw in res[res["after_cleaning"] & ~res["raw_parse"]]["raw_output"].head(3):
    print(repr(raw[:250]), "\n")
print("--- Parse edilemeyen / şema hatası örnekleri ---")
for _, r in res[~res["schema_ok"]].head(3).iterrows():
    print(r["error"], "|", repr(r["raw_output"][:200]), "\n")

exp_fewshot = run_experiment(dev, "schema", few_shot=True, name="dev_few_shot_format_schema")
print(json_report(exp_fewshot, "few-shot, schema"))

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt

LABELS = ["negative", "neutral", "positive"]

def eval_sentiment(res, name):
    v = res.dropna(subset=["pred_label"])
    acc = (v["true_label"] == v["pred_label"]).mean() * 100
    f1 = f1_score(v["true_label"], v["pred_label"], labels=LABELS, average="macro", zero_division=0) * 100
    print(f"\n===== {name} =====  acc={acc:.2f}%  macro-F1={f1:.2f}%  (n={len(v)})")
    print(classification_report(v["true_label"], v["pred_label"], labels=LABELS, zero_division=0))
    return v, acc, f1

v0, acc0, f10 = eval_sentiment(exp_schema,  "DEV zero-shot")
v1, acc1, f11 = eval_sentiment(exp_fewshot, "DEV few-shot")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, v, title in zip(axes, [v0, v1], ["Zero-shot", "Few-shot"]):
    cm = confusion_matrix(v["true_label"], v["pred_label"], labels=LABELS)
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, colorbar=False)
    ax.set_title(f"DEV {title}")
plt.tight_layout()
plt.savefig(OUT_DIR / "dev_confusion_matrices.png", dpi=150)
plt.show()

wrong = v1[v1["true_label"] != v1["pred_label"]].copy()
print("Yanlış tahmin:", len(wrong))
display(wrong.groupby(["true_label", "pred_label"]).size().reset_index(name="count")
        .sort_values("count", ascending=False))

pd.set_option("display.max_colwidth", 130)
display(wrong[["text", "true_label", "pred_label", "sentiment_score"]].head(20))

# Skor sentiment ile uyumlu mu? (polarite: negatif ~0, nötr ~0.5, pozitif ~1)
print("\nOrtalama sentiment_score (gerçek etikete göre):")
print(v1.groupby("true_label")["sentiment_score"].agg(["mean", "std"]).round(3))

import numpy as np

SYSTEM_PROMPT_V2 = f"""You are an information extraction system for airline customer tweets.
Return ONLY one valid JSON object. No markdown, no explanations.

Output format:
{{"category": "positive|negative|neutral", "sentiment_score": 0.0, "entities": [{{"text": "...", "type": "..."}}], "issues": ["..."]}}

Field definitions:
- category: overall sentiment the customer expresses about the airline or its service.
  positive = the customer clearly praises the airline, thanks it for something good, or expresses satisfaction.
  negative = the customer clearly complains, expresses frustration, or reports a service failure
  (delay, cancellation, lost bag, rude staff, etc.).
  neutral = everything else: questions, requests for help or information, requests to change or rebook,
  suggestions or wishes, news, jokes, or tweets with no clear opinion about the airline.
- sentiment_score: polarity from 0.0 (very negative) to 1.0 (very positive); 0.5 means neutral.
- entities: named entities that appear in the tweet. "text" must be copied exactly from the tweet.
  Airline names and @handles have type airline. "type" must be one of {ENTITY_TYPES}.
- issues: problems reported by the customer. Each item must be one of {ISSUE_LABELS}.
  Use [] when no problem is reported.

Rules:
- Judge the whole tweet, not single keywords.
- A question or request for help (flight status, rebooking, changing a reservation, policies, promo codes)
  is neutral, even if it mentions delays, weather or cancellations, unless the customer also complains
  or sounds frustrated.
- Mentioning a problem word (delay, cancelled, late) is not a complaint by itself.
  Look for blame, frustration or dissatisfaction.
- Polite words ("thanks", "please"), emojis, hashtags and jokes do not make a tweet positive.
  Positive requires praise or satisfaction with the airline.
- A suggestion or wish ("you should offer...", "I wish you flew to...") is neutral.
- Do not invent entities or issues that are not in the tweet.
- If there are no entities or issues, use empty lists.
"""

# Ek few-shot örnekleri (uydurma SkyJet; veri setinden alınmadı)
FEW_SHOT_V2 = FEW_SHOT + [
    ("@SkyJet can I change my reservation to the 14th? we got rebooked because of the storm",
     {"category": "neutral", "sentiment_score": 0.5,
      "entities": [{"text": "@SkyJet", "type": "airline"}], "issues": []}),
    ("@SkyJet any delays on the evening flight from Denver to Austin?",
     {"category": "neutral", "sentiment_score": 0.5,
      "entities": [{"text": "@SkyJet", "type": "airline"}, {"text": "Denver", "type": "location"},
                   {"text": "Austin", "type": "location"}], "issues": []}),
    ("@SkyJet you should add more flights to Seattle, I would love that",
     {"category": "neutral", "sentiment_score": 0.5,
      "entities": [{"text": "@SkyJet", "type": "airline"}, {"text": "Seattle", "type": "location"}], "issues": []}),
    ("@SkyJet my bag never arrived and the staff were rude about it",
     {"category": "negative", "sentiment_score": 0.05,
      "entities": [{"text": "@SkyJet", "type": "airline"}], "issues": ["Lost Luggage", "Customer Service Issue"]}),
    ("@SkyJet loved the free wifi on my flight today, great job",
     {"category": "positive", "sentiment_score": 0.92,
      "entities": [{"text": "@SkyJet", "type": "airline"}], "issues": []}),
]
# Örneklerdeki issue etiketleri geçerli mi?
for _, out in FEW_SHOT_V2:
    assert all(i in ISSUE_LABELS for i in out["issues"]), out

def build_prompt(tweet, few_shot=False):
    """few_shot: False = zero-shot (v1 prompt) | True = few-shot v1 | 'v2' = yeni prompt + 8 örnek"""
    system = SYSTEM_PROMPT_V2 if few_shot == "v2" else SYSTEM_PROMPT
    shots = FEW_SHOT_V2 if few_shot == "v2" else (FEW_SHOT if few_shot else [])
    prompt = system
    if shots:
        prompt += "\nExamples:\n"
        for t, out in shots:
            prompt += f"\nTweet: {json.dumps(t)}\nJSON: {json.dumps(out)}\n"
    prompt += f"\nTweet: {json.dumps(tweet, ensure_ascii=False)}\nJSON:"
    return prompt

# Dev üzerinde v2
exp_fewshot_v2 = run_experiment(dev, "schema", few_shot="v2", name="dev_few_shot_v2_schema")

def dev_row(res, name):
    v = res.dropna(subset=["pred_label"])
    rep = classification_report(v["true_label"], v["pred_label"], labels=LABELS,
                                output_dict=True, zero_division=0)
    acc = rep["accuracy"]
    ci = 1.96 * np.sqrt(acc * (1 - acc) / len(v))
    return {"prompt": name, "accuracy_%": round(acc * 100, 2), "±95%CI": round(ci * 100, 1),
            "macro_F1_%": round(rep["macro avg"]["f1-score"] * 100, 2),
            "neg_recall": round(rep["negative"]["recall"], 2),
            "neu_recall": round(rep["neutral"]["recall"], 2),
            "pos_recall": round(rep["positive"]["recall"], 2),
            "json_final_%": round(res["final_ok"].mean() * 100, 2)}

cmp = pd.DataFrame([
    dev_row(exp_schema,       "zero-shot"),
    dev_row(exp_fewshot,      "few-shot v1"),
    dev_row(exp_fewshot_v2,   "few-shot v2"),
])
display(cmp)
cmp.to_csv(OUT_DIR / "dev_prompt_comparison.csv", index=False, encoding="utf-8-sig")

print(classification_report(exp_fewshot_v2.dropna(subset=["pred_label"])["true_label"],
                            exp_fewshot_v2.dropna(subset=["pred_label"])["pred_label"],
                            labels=LABELS, zero_division=0))

VARIANTS = {"zero-shot": False, "few-shot v1": True, "few-shot v2": "v2"}
BEST_NAME = cmp.sort_values("macro_F1_%", ascending=False).iloc[0]["prompt"]
print("Dev'e göre seçilen prompt:", BEST_NAME)
print("Bundan sonra prompt'u DEĞİŞTİRMİYORUZ; test seti tek seferde çalıştırılacak.")

test_base = run_experiment(test, "schema", few_shot=False,               name="test_zero_shot_schema")
test_best = run_experiment(test, "schema", few_shot=VARIANTS[BEST_NAME], name="test_final_schema")

test_table = pd.DataFrame([dev_row(test_base, "zero-shot (baseline)"),
                           dev_row(test_best, f"FINAL: {BEST_NAME}")])
display(test_table)
test_table.to_csv(OUT_DIR / "test_results_table.csv", index=False, encoding="utf-8-sig")

print(pd.DataFrame([json_report(test_base, "test zero-shot"), json_report(test_best, "test final")]).T)

vt = test_best.dropna(subset=["pred_label"])
print(classification_report(vt["true_label"], vt["pred_label"], labels=LABELS, zero_division=0))
cm = confusion_matrix(vt["true_label"], vt["pred_label"], labels=LABELS)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(colorbar=False)
plt.title(f"TEST confusion matrix ({BEST_NAME})")
plt.tight_layout(); plt.savefig(OUT_DIR / "test_confusion_matrix.png", dpi=150); plt.show()

def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def extraction_eval(res, name):
    r = res[res["final_ok"]].copy()
    r["ents"] = r["entities"].apply(json.loads)
    r["iss"]  = r["issues"].apply(json.loads)
    print(f"\n===== EXTRACTION: {name} (n={len(r)}) =====")

    # 1) Havayolu varlığı: gold = 'airline' sütunu (yalnızca tweette geçtiği durumlar)
    m = r[r.apply(lambda x: norm(x["airline"]) in norm(x["text"]), axis=1)]
    hit_any   = m.apply(lambda x: any(norm(x["airline"]) in norm(e["text"]) for e in x["ents"]), axis=1)
    hit_typed = m.apply(lambda x: any(norm(x["airline"]) in norm(e["text"]) and e["type"] == "airline"
                                      for e in x["ents"]), axis=1)
    print(f"Havayolu tweette geçen örnek: {len(m)}/{len(r)}")
    print(f"  Havayolu bulundu (herhangi tip): {hit_any.mean()*100:.1f}%")
    print(f"  Havayolu bulundu ve tipi 'airline': {hit_typed.mean()*100:.1f}%")

    # 2) Halüsinasyon: entity metni tweette birebir geçiyor mu?
    rows = [(x["text"], e["text"], e["type"]) for _, x in r.iterrows() for e in x["ents"]]
    grounded = [e_txt.lower() in t.lower() for t, e_txt, _ in rows]
    print(f"Toplam entity: {len(rows)} (tweet başına {len(rows)/len(r):.2f})")
    print(f"  Tweette birebir geçen (grounded): {np.mean(grounded)*100:.1f}%  "
          f"| halüsinasyon: {(1-np.mean(grounded))*100:.1f}%")
    ungrounded = [(t, e, ty) for (t, e, ty), g in zip(rows, grounded) if not g]
    for t, e, ty in ungrounded[:8]:
        print(f"    ✗ '{e}' ({ty})  <-  {t[:80]!r}")

    # 3) Issues: gold = negativereason ('Can't Tell' hariç), yalnızca negatif tweetler
    neg = r[(r["true_label"] == "negative") & r["negativereason"].notna()
            & (r["negativereason"] != "Can't Tell")].copy()
    neg["hit"] = neg.apply(lambda x: x["negativereason"] in x["iss"], axis=1)
    print(f"\nIssues (negatif, gold etiketli n={len(neg)}): gold etiket tahmin listesinde: {neg['hit'].mean()*100:.1f}%")
    display(neg.groupby("negativereason")["hit"].agg(hit_rate="mean", n="count").round(2))
    fp = r[r["true_label"] != "negative"]["iss"].apply(len).gt(0).mean()
    print(f"Negatif OLMAYAN tweetlerde issue üretme oranı (yanlış alarm): {fp*100:.1f}%")

    return {"airline_found_%": round(hit_any.mean()*100, 1), "airline_typed_%": round(hit_typed.mean()*100, 1),
            "entity_grounded_%": round(np.mean(grounded)*100, 1),
            "issue_gold_hit_%": round(neg["hit"].mean()*100, 1),
            "issue_false_alarm_%": round(fp*100, 1)}

ext_summary = extraction_eval(test_best, f"test / {BEST_NAME}")
with open(OUT_DIR / "extraction_summary.json", "w", encoding="utf-8") as f:
    json.dump(ext_summary, f, indent=2)